In [0]:
%sql
-- Criando o catálogo pra organizar tudo do projeto e não misturar os dados
CREATE CATALOG IF NOT EXISTS medalhao

In [0]:
%sql 
-- Tudo o que vier depois estará dentro do catálogo 'medalhao'
USE CATALOG medalhao;

-- Criando a pasta 'bronze' se ela ainda não existir, pra eu ter onde jogar os dados brutos
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
# Trazendo as funções do Spark pra tratar as colunas e as datas depois
from pyspark.sql import functions as F

# Guardando as pastas em variáveis, caso mude um nome no futuro, não precisar caçar erro no código todo
catalogo = "medalhao"
schema = "bronze"
volume = "olist_raw"

# Montando o caminho de onde os arquivos CSV estão salvos
volume_path = f"/Volumes/{catalogo}/{schema}/{volume}"

In [0]:
# Essa função foi feita pra automatizar a carga e não ter que ficar repetindo código pros 9 arquivos
def ingest_csv(nome_arquivo, nome_tabela):
    try:
        print(f"Iniciando a ingestão do arquivo{nome_arquivo}")

#Lendo o CSV: avisando que tem cabeçalho e pedindo pro Spark tratar os tipos de dados sozinho
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(f"{volume_path}/{nome_arquivo}")

# Criando a coluna de tempo pra saber exatamente quando esse dado entrou no sistema
        df_final = df.withColumn("timestamp_ingestion", F.current_timestamp())

# Salvando na Bronze, se a tabela já existir, ele sobrescreve com os dados novos
        df_final.write.mode("overwrite").saveAsTable(f"{schema}.{nome_tabela}")

        print(f"Tabela {schema}.{nome_tabela} criada com sucesso!")

#ele me avisa aqui sem travar o resto do código, caso de algum erro no codigo
    except Exception as e:
        print(f"Erro ao processar {nome_arquivo}: {str(e)}")







In [0]:
# Chamei a função passando os arquivos "csvs" e o nome da tabela para rodar
ingest_csv("olist_customers_dataset.csv", "tb_customers")
ingest_csv("olist_geolocation_dataset.csv", "tb_geolocalizacao")
ingest_csv("olist_order_items_dataset.csv", "tb_order_items")
ingest_csv("olist_order_payments_dataset.csv","tb_order_payments")
ingest_csv("olist_order_reviews_dataset.csv", "tb_order_reviews")
ingest_csv("olist_orders_dataset.csv", "tb_orders")
ingest_csv("olist_products_dataset.csv", "tb_products")
ingest_csv("olist_sellers_dataset.csv", "tb_sellers")
ingest_csv("product_category_name_translation.csv", "tb_product_category_name_translation") 

Iniciando a ingestão do arquivoolist_customers_dataset.csv
Tabela bronze.tb_customers criada com sucesso!
Iniciando a ingestão do arquivoolist_geolocation_dataset.csv
Tabela bronze.tb_geolocalizacao criada com sucesso!
Iniciando a ingestão do arquivoolist_order_items_dataset.csv
Tabela bronze.tb_order_items criada com sucesso!
Iniciando a ingestão do arquivoolist_order_payments_dataset.csv
Tabela bronze.tb_order_payments criada com sucesso!
Iniciando a ingestão do arquivoolist_order_reviews_dataset.csv
Tabela bronze.tb_order_reviews criada com sucesso!
Iniciando a ingestão do arquivoolist_orders_dataset.csv
Tabela bronze.tb_orders criada com sucesso!
Iniciando a ingestão do arquivoolist_products_dataset.csv
Tabela bronze.tb_products criada com sucesso!
Iniciando a ingestão do arquivoolist_sellers_dataset.csv
Tabela bronze.tb_sellers criada com sucesso!
Iniciando a ingestão do arquivoproduct_category_name_translation.csv
Tabela bronze.tb_product_category_name_translation criada com suce

In [0]:
# Criando as caixinhas de texto no topo do notebook pra alterar as datas da consulta sem mexer no código
dbutils.widgets.text("data_inicio", "01-01-2023") 
dbutils.widgets.text("data_fim", "12-31-2023")

In [0]:
# Importando as bibliotecas para realizar a requisição HTTP (requests), manipulação de dados em memória (pandas) e funções do Spark
import requests
import pandas as pd
from pyspark.sql import functions as F

# Definindo o período de 2016 a 2018 para garantir que a cotação cubra todo o dataset da Olist
data_ini = "01-01-2016" 
data_fim = "12-31-2018"
schema = "medalhao.bronze" 

# Montando a URL dinâmica para bater no endpoint do Banco Central (API) utilizando os parâmetros de data inicial e final
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_ini}'&@dataFinalCotacao='{data_fim}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

try:
    # Realizando a chamada na API e extraindo a lista de valores do JSON retornado
    response = requests.get(url)
    data = response.json()['value'] 
    
    # Criando um DataFrame em Pandas como ponte para converter os dados da API para o formato de DataFrame do Spark
    df_pd = pd.DataFrame(data)
    df_spark = spark.createDataFrame(df_pd)
    
    # Realizando o mapeamento da bronze: convertendo a string de data da API para o tipo Timestamp. O withColumn também adiciona o tempo da ingestão para controle de auditoria dos dados brutos
    df_bronze_cotacao = (
        df_spark
        .withColumn("dataHoraCotacao", F.to_timestamp("dataHoraCotacao"))
        .withColumn("timestamp_ingestion", F.current_timestamp())
    )
    
    # Salvando os dados brutos na camada Bronze no formato de tabela Delta
    df_bronze_cotacao.write.mode("overwrite").saveAsTable(f"{schema}.tb_cotacao_dolar")

except Exception as e:
    # Tratamento de erro básico caso a API esteja fora do ar ou o formato do JSON mude
    print(f"Erro na ingestão da API: {e}")